# Notebook 01 — ICMPRS Data Exploration

This notebook verifies the parametric cohort generation and reproduces the calibration checks from **Table III** of the manuscript.

> **Note:** All data used here is synthetic. Results reflect the generator's distributional properties, not clinical ground truth.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from icmprs.generator import ICMPRSGenerator
from evaluation.metrics import mmd_squared

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
SEED = 42

## 1. Generate the primary cohort

In [ ]:
gen = ICMPRSGenerator(seed=SEED)
df  = gen.generate(n=1995)

print(f'Shape: {df.shape}')
print(f'PD: {(df.label==1).sum()}  HC: {(df.label==0).sum()}')
df.head(3)

## 2. Cohort composition (Table, manuscript Section III-C)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col, title in zip(axes,
                           ['dialect', 'device', 'footwear'],
                           ['Dialect Region', 'Device Tier', 'Footwear']):
    vc = df[col].value_counts(normalize=True) * 100
    vc.plot(kind='bar', ax=ax, color=sns.color_palette('muted'))
    ax.set_title(title)
    ax.set_ylabel('Percentage (%)')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig('../figures/cohort_composition.pdf', bbox_inches='tight')
plt.show()
print('Saved figures/cohort_composition.pdf')

## 3. Indian-specific feature distributions (PD vs HC)

In [ ]:
indian_features = ['rca_index', 'nre_vli', 'dpld_lift_duration',
                   'ajhi_transition_time', 'bsv']
labels_map = {1: 'PD', 0: 'HC'}
df['group'] = df.label.map(labels_map)

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
titles = ['RCA Index (Hz)', 'NRE-VLI', 'DPLD Lift Duration (ms)',
          'AJHI Transition Time (ms)', 'BSV']

for ax, feat, title in zip(axes, indian_features, titles):
    for label, colour in [(1, '#e06c75'), (0, '#61afef')]:
        ax.hist(df.loc[df.label==label, feat],
                bins=30, alpha=0.6, label=labels_map[label],
                color=colour, density=True)
    ax.set_title(title, fontsize=9)
    ax.legend(fontsize=8)
    ax.set_xlabel('')

plt.suptitle('Indian-Specific Feature Distributions (Synthetic Benchmark)',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig('../figures/indian_feature_distributions.pdf', bbox_inches='tight')
plt.show()

## 4. Calibration check — Table III

In [ ]:
# Published clinical reference values (Table III-A)
reference = {
    'jitter_local':  {'real_pd': 0.031, 'source': 'Little 2009'},
    'shimmer_local': {'real_pd': 0.167, 'source': 'Tsanas 2012'},
    'hnr':           {'real_pd': 14.1,  'source': 'Harel 2004'},
    'stride_length': {'real_pd': 0.71,  'source': 'Wahid 2015'},
    'stride_speed':  {'real_pd': 0.68,  'source': 'Zeng 2016'},
}

rows = []
for feat, ref in reference.items():
    if feat not in df.columns:
        continue
    mu_pd = df.loc[df.label==1, feat].mean()
    mu_hc = df.loc[df.label==0, feat].mean()
    rows.append({
        'Feature': feat,
        'ICMPRS PD': round(mu_pd, 4),
        'ICMPRS HC': round(mu_hc, 4),
        'Real PD': ref['real_pd'],
        'Source': ref['source'],
        'Within 1 SD': abs(mu_pd - ref['real_pd']) < df.loc[df.label==1, feat].std()
    })

cal_df = pd.DataFrame(rows)
print(cal_df.to_string(index=False))

## 5. Label-permutation sanity check

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

X = df[gen.FEATURE_NAMES_ACOUSTIC].values
y_true = df.label.values
y_perm = np.random.default_rng(42).permutation(y_true)

scaler = StandardScaler()
X_s    = scaler.fit_transform(X)

svc = SVC(kernel='rbf', C=10, gamma=0.01, random_state=42, class_weight='balanced')

acc_true = cross_val_score(svc, X_s, y_true, cv=5, scoring='accuracy').mean()
acc_perm = cross_val_score(svc, X_s, y_perm, cv=5, scoring='accuracy').mean()

print(f'True labels accuracy:      {acc_true*100:.1f}%')
print(f'Permuted labels accuracy:  {acc_perm*100:.1f}%  (expect ~50%)')
assert acc_perm < 0.55, 'Permutation check failed — possible data leakage!'